# 09 · Transfer learning — Oxford Flowers102

Scratch CNN vs frozen ResNet18 + new head.

Source: `CNN_Exercise/Transfer_Learning_Exercise.ipynb`

**Colab:** Runtime → GPU. First run downloads Flowers102 (~300 MB). Scratch CNN on 102 classes will look weak; that is the point of the comparison.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_dataset = datasets.Flowers102(root='data', split='train', download=True, transform=transform)
val_dataset = datasets.Flowers102(root='data', split='val', download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
print('train', len(train_dataset), 'val', len(val_dataset), 'classes 102')

Scratch CNN: two conv blocks (32 then 64), feature map 64×56×56 after two 2×2 pools on 224 input.

In [ ]:
class FlowerCNN(nn.Module):
    def __init__(self, num_classes=102):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Flatten(),
        )
        self.classifier = nn.Sequential(
            nn.Linear(64 * 56 * 56, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model_scratch = FlowerCNN().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_scratch.parameters(), lr=0.0001)
print(model_scratch)

In [ ]:
def train_model_scratch(model, loader, loss_fn, optimizer, num_epochs=8):
    model.train()
    for epoch in range(num_epochs):
        running_loss = correct = total = 0
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        print(f'Epoch [{epoch+1}/{num_epochs}], '
              f'Loss: {running_loss/len(loader):.4f}, '
              f'Accuracy: {100*correct/total:.2f}%')

train_model_scratch(model_scratch, train_loader, loss_fn, optimizer, num_epochs=8)

In [ ]:
def evaluate_model(model, loader, tag='Test'):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    print(f'{tag} Accuracy: {100 * correct / total:.2f}%')

evaluate_model(model_scratch, val_loader, 'Scratch CNN val')

## Transfer learning — freeze ResNet18, train the new 102-way head

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
for param in model.parameters():
    param.requires_grad = False

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 102)
model = model.to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

In [ ]:
def train_model(model, loader, loss_fn, optimizer, num_epochs=6):
    model.train()
    for epoch in range(num_epochs):
        running_loss = correct = total = 0
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        print(f'Epoch [{epoch+1}/{num_epochs}], '
              f'Loss: {running_loss/len(loader):.4f}, '
              f'Accuracy: {100*correct/total:.2f}%')

train_model(model, train_loader, loss_fn, optimizer, num_epochs=6)
evaluate_model(model, val_loader, 'ResNet18 transfer val')